# Modeling ประมาณองค์ประกอบร่างกายจากค่าวัด

ใช้ชุดข้อมูลจาก `src/real_label.py` (NHANES 2017-2018 ผู้ใหญ่ 18-59 ปีที่มีผล DXA ครบ) ประมาณ Fat % กับมวลกล้ามเนื้อ (Lean mass, ALM, Trunk lean) จากค่าที่วัดง่าย โดยไม่ต้องสแกน DXA แล้วต่อยอดไปคัดกรองคนที่มีมวลกล้ามเนื้อต่ำ

In [1]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold, cross_val_predict, cross_val_score, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier
warnings.filterwarnings("ignore")

os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
sys.path.insert(0, "src")
import real_label as rl
df, n_adults = rl.build()
df.to_csv(rl.OUT, index=False)
BASIC, CIRC, LIFE = rl.BASIC, rl.CIRC, rl.LIFE
FEAT = BASIC + CIRC
TARGETS = ["FAT_PCT", "LEAN_KG", "ALM_KG", "TRUNK_KG"]
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df.LOW_HT2)
print(f"adults 18-59: {n_adults} -> with DXA + body measures: {len(df)} | train {len(train)} / test {len(test)}")

def score(y, p):
    return {"R2": r2_score(y, p), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p)))}

def ridge():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=[0.1, 1, 10, 100]))

summary = {"n_adults": n_adults, "n": len(df), "n_train": len(train), "n_test": len(test),
           "prevalence": {"LOW_HT2": float(df.LOW_HT2.mean()), "LOW_FNIH": float(df.LOW_FNIH.mean()), "both": int((df.LOW_FNIH & df.LOW_HT2).sum())}}

adults 18-59: 3515 -> with DXA + body measures: 2310 | train 1848 / test 462
